In [1]:
import pandas as pd

In [2]:
df= pd.read_csv('../data/raw/AB Testing Data.csv') 

In [4]:
df.head(10)

,user_id,timestamp,group,landing_page,converted,age,gender,location,session_duration,pages_visited,device_type,purchase_amount
0,U1,2025-08-02 15:27:54.137058,control,old_page,0,37,Male,Pakistan,3.69,4,Mobile,0.0
1,U2,2024-04-22 10:22:51.712050,treatment,new_page,0,31,Female,UK,1.29,3,Desktop,0.0
2,U3,2024-08-14 21:35:11.135894,treatment,new_page,0,38,Male,US,3.72,5,Desktop,0.0
3,U4,2025-03-19 03:28:51.120807,treatment,new_page,0,28,Female,India,7.76,2,Mobile,0.0
4,U5,2024-12-22 13:13:17.973162,control,old_page,0,33,Male,Australia,6.78,6,Mobile,0.0
5,U6,2025-01-02 06:02:32.511149,control,old_page,0,30,Male,US,2.88,1,Desktop,0.0
6,U7,2024-01-30 07:31:09.175272,control,old_page,0,32,Male,UK,5.39,3,Desktop,0.0
7,U8,2025-05-07 22:29:39.247674,treatment,new_page,0,34,Male,India,5.39,7,Desktop,0.0
8,U9,2025-08-22 13:42:17.098595,treatment,new_page,0,48,Female,Pakistan,4.99,3,Mobile,0.0
9,U10,2024-09-13 12:38:48.131095,treatment,new_page,0,27,Female,Australia,6.82,7,Desktop,0.0


In [5]:
df.columns

Index(['user_id', 'timestamp', 'group', 'landing_page', 'converted', 'age',
       'gender', 'location', 'session_duration', 'pages_visited',
       'device_type', 'purchase_amount'],
      dtype='str')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 294478 entries, 0 to 294477
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   user_id           294478 non-null  str    
 1   timestamp         294478 non-null  str    
 2   group             294478 non-null  str    
 3   landing_page      294478 non-null  str    
 4   converted         294478 non-null  int64  
 5   age               294478 non-null  int64  
 6   gender            294478 non-null  str    
 7   location          294478 non-null  str    
 8   session_duration  294478 non-null  float64
 9   pages_visited     294478 non-null  int64  
 10  device_type       294478 non-null  str    
 11  purchase_amount   294478 non-null  float64
dtypes: float64(2), int64(3), str(7)
memory usage: 45.1 MB


In [7]:
df.shape

(294478, 12)

In [9]:
df['user_id'].duplicated().sum()

np.int64(0)

In [10]:
df['group'].value_counts()

group
treatment    147552
control      146926
Name: count, dtype: int64

In [11]:
df['landing_page'].value_counts()   

landing_page
new_page    147552
old_page    146926
Name: count, dtype: int64

In [13]:
df['device_type'].value_counts()

device_type
Desktop    176692
Mobile     103093
Tablet      14693
Name: count, dtype: int64

In [14]:
df['converted'].value_counts()

converted
0    250550
1     43928
Name: count, dtype: int64

In [15]:
df['converted'].value_counts(normalize=True) * 100

converted
0    85.082757
1    14.917243
Name: proportion, dtype: float64

In [16]:
df.describe()   

,converted,age,session_duration,pages_visited,purchase_amount
count,294478.000000,294478.000000,294478.000000,294478.000000,294478.000000
mean,0.149172,31.902084,5.002136,4.019550,5.610490
std,0.356259,9.250043,1.980285,1.965182,15.438006
min,0.000000,18.000000,0.500000,1.000000,0.000000
25%,0.000000,25.000000,3.640000,3.000000,0.000000
50%,0.000000,32.000000,4.990000,4.000000,0.000000
75%,0.000000,38.000000,6.340000,5.000000,0.000000
max,1.000000,65.000000,13.730000,18.000000,269.280000


In [17]:
df.groupby('group')['converted'].aggregate(['mean', 'sum', 'count'])

,mean,sum,count
group,,,
control,0.118726,17444,146926
treatment,0.179489,26484,147552


STATISTICS NOTES

A/B TESTING

Control group:
The group assigned to the old page.

Treatment group:
The group assigned to the new page.

Converted:
1 = the user converted/purchased.
0 = the user did not convert/purchase.

Our experiment:
We compare users who saw the old page with users who saw
the new page and examine their conversion rates.

Null hypothesis (H₀):
There is no difference in conversion rates between the old
and new pages.

Alternative hypothesis (H₁):
There is a difference in conversion rates between the old
and new pages.

In [3]:
from statsmodels.stats.proportion import proportions_ztest

successes = [
    df[df['group'] == 'control']['converted'].sum(),
    df[df['group'] == 'treatment']['converted'].sum()
]

samples = [
    (df['group'] == 'control').sum(),
    (df['group'] == 'treatment').sum()
]

z_stat, p_value = proportions_ztest(successes, samples)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

Z-statistic: -46.277339891305736
P-value: 0.0


In [5]:
control_conversions = 17444
control_users = 146926

treatment_conversions = 26484
treatment_users = 147552

p_control = control_conversions / control_users
p_treatment = treatment_conversions / treatment_users   

print("Control Conversion Rate:", p_control)
print("Treatment Conversion Rate:", p_treatment)

Control Conversion Rate: 0.11872643371493133
Treatment Conversion Rate: 0.1794892648015615


In [6]:
import math

pooled_rate = (
    control_conversions + treatment_conversions
) / (
    control_users + treatment_users
)

standard_error = math.sqrt(
    pooled_rate * (1 - pooled_rate) *
    (1 / control_users + 1 / treatment_users)
)

print("Pooled rate:", pooled_rate)
print("Standard error:", standard_error)

Pooled rate: 0.14917243393394414
Standard error: 0.0013130147763321604


In [7]:
z_manual = (p_control - p_treatment) / standard_error

print("Manual Z-statistic:", z_manual)

Manual Z-statistic: -46.277339891305736


In [9]:

# Calculate the difference in conversion rates

difference = p_treatment - p_control

print("Difference:", difference)
print("Percentage point increase:", difference * 100)

Difference: 0.060762831086630165
Percentage point increase: 6.0762831086630165


In [10]:
se_ci = math.sqrt(
    (p_control * (1 - p_control) / control_users) +
    (p_treatment * (1 - p_treatment) / treatment_users)
)

print("Standard Error for CI:", se_ci)

Standard Error for CI: 0.0013077609211911286
